# Genome Repair Console -- Post-Stage-0: Evaluation & Benchmarking

**Does this need a GPU? No, for most of it.** `evaluate.py` and `variant_preservation_test.py` are inference-only (no backward pass) -- and this model's own architecture is bottlenecked by its sequential, per-step autoregressive decode loop, not raw matmul throughput (train.py prints this itself: *"per-batch cost scales with chunk_size... largely independent of GPU vs CPU"*). So CPU isn't the dramatic step down it would be for a typical model. Part B (the full Racon/Medaka baseline comparison) is also CPU-native for Flye and Racon; Medaka works on CPU, just slower. **Your 16-20GB RAM is comfortably enough for a single ~4.6Mbp bacterial genome** -- this workload is nowhere near the scale where that would be tight.

**Structure:**
- **Part A** (below): `evaluate.py` on a fresh held-out set + `variant_preservation_test.py`. Self-contained, reuses tools already installed, no new external downloads. Do this first.
- **Part B**: the full Flye -> Racon -> Medaka baseline comparison via `run_comparison.sh`. Needs several new tools installed (flye, racon, medaka, pomoxis, mummer4, rasusa) and is meaningfully heavier/more failure-prone -- review Part A's results before committing to this.

If running on Kaggle: same GPU-off-by-default is fine here (Accelerator -> None saves your GPU quota for actual training later) -- **or run this locally** per the guidance at the very end of this notebook.

## 1. Environment setup -- same pattern as the Stage 0 notebook

In [ ]:
!apt-get update -qq && apt-get install -y -qq minimap2
!minimap2 --version


In [ ]:
import importlib
def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
        print(f'{pkg}: already available')
    except ImportError:
        get_ipython().system(f'pip install -q {pip_name or pkg}')

ensure('edlib')
ensure('mappy')
ensure('fastapi')


In [ ]:
!pip show badread 2>/dev/null | grep -q 'Version: 0.4.2' && echo 'badread 0.4.2 already installed' || pip install -q git+https://github.com/rrwick/Badread.git@v0.4.2
!badread --version


## 2. Copy your code (with the trained checkpoint included)

This time your uploaded dataset needs to also include `checkpoints/model_best.pt` (the epoch-2 best checkpoint) -- download it from Kaggle if you haven't already, add it to your code dataset, and re-upload a new version before running this.

In [ ]:
CODE_DATASET_SLUG = 'YOUR-CODE-DATASET-SLUG-WITH-CHECKPOINT'  # <-- EDIT THIS, verify with !ls /kaggle/input/ first

import shutil, os
src = f'/kaggle/input/{CODE_DATASET_SLUG}'
dst = '/kaggle/working/final-year-project'

if os.path.exists(dst):
    print('Project already copied, skipping (delete /kaggle/working/final-year-project to force a fresh copy).')
else:
    shutil.copytree(src, dst)
    print(f'Copied {src} -> {dst}')

%cd /kaggle/working/final-year-project
!ls checkpoints/ data/reference/


## 3. Sanity gate

In [ ]:
!find . -name '*.py' -exec python -m py_compile {} \;
!python -m pytest tests/ -q


## 4. Confirm the checkpoint and both reference genomes are present

You need BOTH: K-12 MG1655 (training substrate) and the Zymo benchmark strain (for anything that touches real-strain comparison later).

In [ ]:
from pathlib import Path

checkpoint = Path('checkpoints/model_best.pt')
k12_ref = Path('data/reference/ecoli_k12_mg1655.fasta')
zymo_ref = Path('data/reference/ecoli_zymo_benchmark_strain.fasta')

assert checkpoint.exists(), 'checkpoints/model_best.pt not found -- did you include it in your code dataset?'
assert k12_ref.exists(), 'K-12 reference not found'
assert zymo_ref.exists(), 'Zymo benchmark reference not found -- needed for Part B'

print(f'{checkpoint}: {checkpoint.stat().st_size:,} bytes -- OK')
print(f'{k12_ref}: {k12_ref.stat().st_size:,} bytes -- OK')
print(f'{zymo_ref}: {zymo_ref.stat().st_size:,} bytes -- OK')


## PART A.1 -- Real accuracy numbers: `evaluate.py`

`evaluate.py` needs a JSONL of ground-truth pairs it wasn't trained on. We generate a **fresh** set here with a different seed than training used -- honest framing: this evaluates generalization to new noise instantiations of already-seen genomic content, the same thing train.py's own internal validation split already measures, just independently and with the full free-running (no teacher forcing) pipeline + real bioinformatics metrics. It is **not** a cross-strain test -- that's what the Zymo comparison in Part B is for.

Kept intentionally small (`--quantity 2x`, `--max-examples 100`) so this finishes in a reasonable time; increase both once you've confirmed the pipeline works.

In [ ]:
import subprocess

subprocess.run([
    'python', '-m', 'data.simulator',
    '--reference', 'data/reference/ecoli_k12_mg1655.fasta',
    '--output', 'data/eval_pairs.jsonl',
    '--quantity', '2x',
    '--seed', '999',  # different from training's seed=42, so these are genuinely different simulated reads
], check=True)


In [ ]:
subprocess.run([
    'python', '-u', '-m', 'training.evaluate',
    '--checkpoint', 'checkpoints/model_best.pt',
    '--validation-data', 'data/eval_pairs.jsonl',
    '--output-dir', 'evaluation_output',
    '--max-examples', '100',
], check=True)


**What to look at:** `evaluation_output/evaluation_summary.json` -- specifically `mean_identity`, `fraction_frame_preserved`, and `minimap2_mean_identity` (the independent, externally-verified number). `evaluation_output/evaluation_outputs.fasta` has every individual prediction if you want to inspect specific examples by hand.

## PART A.2 -- The most important number: `variant_preservation_test.py`

This is the reference-hallucination measurement -- fully self-contained, no external data needed, plants real mutations into K-12 and checks whether the model preserves them or reverts them to the training reference. `--num-chunks 50 --mutations-per-chunk 3` matches the project's own documented default; raise `--num-chunks` for a tighter confidence interval once you've confirmed this runs cleanly.

In [ ]:
subprocess.run([
    'python', '-u', '-m', 'training.variant_preservation_test',
    '--checkpoint', 'checkpoints/model_best.pt',
    '--reference', 'data/reference/ecoli_k12_mg1655.fasta',
    '--num-chunks', '50',
    '--mutations-per-chunk', '3',
], check=True)


**What to look at:** the printed `variant_preservation_rate` -- closer to 1.0 means the model trusts genuine variants; closer to 0.0 means it's reverting them to the reference (the hallucination failure mode). Given how much of this project's honest positioning rests on this specific number, it's worth treating as at least as important as Part A.1's accuracy figures, maybe more.

---
## PART B (optional, heavier) -- Full baseline comparison: ours vs. Racon/Medaka

Review Part A's results before running this. This installs several new bioinformatics tools and runs a full assembly pipeline three times (uncorrected / Racon+Medaka / ours) -- more moving parts, more that can go wrong, and meaningfully longer runtime than Part A.

### B.1 -- Install the comparison tools

In [ ]:
# Racon -- CPU-native, build from source
!git clone -q https://github.com/lbcb-sci/racon.git --recursive
!cd racon && mkdir -p build && cd build && cmake -DCMAKE_BUILD_TYPE=Release .. -Wno-dev > /dev/null && make -j$(nproc) > /dev/null
import os
os.environ['PATH'] = os.path.abspath('racon/build/bin') + ':' + os.environ['PATH']
!racon --version


In [ ]:
# Medaka -- CPU wheel, avoids pulling unnecessary CUDA binaries
!pip install -q medaka --extra-index-url https://download.pytorch.org/whl/cpu
!medaka --version


In [ ]:
# pomoxis -- ONT's own assessment suite (assess_assembly, assess_homopolymers)
!pip install -q pomoxis
!assess_assembly --help > /dev/null && echo 'pomoxis OK'


In [ ]:
# mummer4 (dnadiff) and rasusa (coverage-normalized downsampling) -- conda is simpler than building from source
!conda install -y -c bioconda mummer4 rasusa -q 2>&1 | tail -5
!dnadiff --version 2>&1 | head -1
!rasusa --version


In [ ]:
# Flye -- assembler used in all three pipeline comparisons
!pip install -q flye
!flye --version


### B.2 -- Get Zymo E. coli reads: real (from ENA) if possible, synthetic as a safe fallback

This cell tries to resolve the real download URL for `ERR7287988` via ENA's own filereport API (rather than guessing the FTP folder-numbering pattern, which varies and I can't fully verify without testing it live). If that fails for any reason -- network restrictions, API changes, accession issues -- it falls back to generating synthetic Zymo-strain reads the same way this project already generates its training data, so Part B can still run end-to-end either way.

In [ ]:
import subprocess, os

ena_url_result = subprocess.run(
    ['curl', '-s', 'https://www.ebi.ac.uk/ena/portal/api/filereport?accession=ERR7287988&result=read_run&fields=fastq_ftp&format=tsv'],
    capture_output=True, text=True
)
print('ENA API response:')
print(ena_url_result.stdout)

ftp_url = None
lines = [l for l in ena_url_result.stdout.strip().split('\n') if l and not l.startswith('run_accession')]
if lines:
    candidate = lines[0].split('\t')[-1].split(';')[0].strip()
    if candidate.startswith('ftp.'):
        ftp_url = 'https://' + candidate

os.makedirs('benchmark_data', exist_ok=True)

if ftp_url:
    print(f'\nResolved real download URL: {ftp_url}')
    subprocess.run(['wget', '-q', '-O', 'benchmark_data/zymo_d6300_raw.fastq.gz', ftp_url], check=True)
    print('Downloaded real Zymo D6300 R10.4.1 reads (whole mock community -- prepare_zymo_subset.sh extracts just E. coli).')
    USE_REAL_ZYMO = True
else:
    print('\nCould not resolve a real download URL automatically.')
    print('Manual option: search ERR7287988 / PRJEB29504 at https://www.ebi.ac.uk/ena/browser/ and download from there,')
    print('then place it at benchmark_data/zymo_d6300_raw.fastq.gz and re-run this cell to pick it up.')
    print('Falling back to SYNTHETIC Zymo-strain reads for now, so Part B can still run end-to-end.')
    USE_REAL_ZYMO = False


In [ ]:
if not USE_REAL_ZYMO:
    subprocess.run([
        'python', '-m', 'data.simulator',
        '--reference', 'data/reference/ecoli_zymo_benchmark_strain.fasta',
        '--output', 'benchmark_data/zymo_synthetic_pairs.jsonl',
        '--quantity', '50x',
        '--seed', '2026',
    ], check=True)
    # convert to a plain FASTQ (fake quality scores) since run_comparison.sh expects FASTQ, not JSONL pairs
    import json
    with open('benchmark_data/zymo_synthetic_pairs.jsonl') as f_in, open('benchmark_data/subset_50x.fastq', 'w') as f_out:
        for i, line in enumerate(f_in):
            rec = json.loads(line)
            seq = rec['noisy_sequence']
            f_out.write(f'@synthetic_read_{i}\n{seq}\n+\n{"I" * len(seq)}\n')
    print('Synthetic subset written to benchmark_data/subset_50x.fastq -- skipping prepare_zymo_subset.sh (only needed for real, multi-species raw data).')


### B.3 -- If using REAL Zymo reads: extract E. coli + downsample

Skip this cell if the previous step fell back to synthetic reads (`USE_REAL_ZYMO == False`) -- they're already single-species and don't need extraction.

In [ ]:
if USE_REAL_ZYMO:
    !chmod +x benchmarking/prepare_zymo_subset.sh
    subprocess.run([
        './benchmarking/prepare_zymo_subset.sh',
        'benchmark_data/zymo_d6300_raw.fastq.gz',
        'data/reference/ecoli_zymo_benchmark_strain.fasta',
        'benchmark_data',
        '50',
        '4.8m',
    ], check=True)
else:
    print('Skipped -- using the synthetic subset already prepared in B.2.')


### B.4 -- Run the full comparison

Uncorrected Flye vs. Flye->Racon(x4)->Medaka vs. our model->Flye (no post-assembly polishing). This is the longest-running cell in this notebook by far -- real assembly + real polishing, three times.

In [ ]:
!chmod +x benchmarking/run_comparison.sh
subprocess.run([
    './benchmarking/run_comparison.sh',
    'benchmark_data/subset_50x.fastq',
    'data/reference/ecoli_zymo_benchmark_strain.fasta',
    'checkpoints/model_best.pt',
    'benchmark_data/results',
], check=True)


**What to look at:** `benchmark_data/results/assessments/*_assess.summary` (Q-scores, indel/mismatch breakdown for each of the three pipelines) and `*_homopolymers` output specifically -- that's the number that validates or invalidates the RLE-channel design decision. Compare your `ours_*` numbers against `racon_medaka_*` directly; per the project's own documented targets, beating a ~39:1-40:1 indel:mismatch ratio (driven by homopolymer improvement specifically) is the bar that matters most.

## Running this locally instead of Kaggle/Colab

**Nothing in Part A needs a GPU** -- `torch.device('cuda' if torch.cuda.is_available() else 'cpu')` is already the default throughout this codebase, so on a machine with no GPU it'll just use CPU automatically, no flags needed. Given the model's own sequential-decode bottleneck (explained at the top of this notebook), don't expect CPU inference to be dramatically slower than GPU here the way CPU *training* would be.

**Part B is also GPU-free** -- Flye and Racon are CPU-native; Medaka runs on CPU (the `--extra-index-url` in B.1 specifically installs Medaka's CPU wheel to avoid pulling in CUDA dependencies you don't need).

**Your 16-20GB RAM is comfortably sufficient** for a single ~4.6Mbp bacterial genome at this scale -- this isn't the workload class where that would be tight.

**To run locally:** clone/copy your project folder, install the same tools as sections 1 and B.1 above via your local package manager (conda is the path of least resistance for `mummer4`/`rasusa`/`pomoxis` outside a Kaggle/Colab image), then run the same `python -m ...` commands directly in your own terminal instead of via `subprocess.run(...)` -- the underlying commands are identical either way, this notebook's `subprocess.run` calls exist only to get clean cell-by-cell output, not because anything here is Kaggle-specific.